### LingoBot Healthcare Assistant - Colab Implementation

This notebook provides a complete implementation of LingoBot that can run in Google Colab with pre-computed indices and data.

#### Setup and Installation


In [2]:
!pip install -U transformers accelerate bitsandbytes torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.0 MB/s eta 0:00:00


In [1]:
%pip install pandas numpy scikit-learn sentence-transformers faiss-cpu gradio ipywidgets


from google.colab import drive
drive.mount('/content/drive')

import sys
import os
from pathlib import Path
import torch

# Add project path to system path
project_path = "/content/drive/MyDrive/lingo/task1"
sys.path.append(project_path)

print("Packages installed and paths configured!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.1 MB/s eta 0:00:00
Mounted at /content/drive
Packages installed and paths configured!


In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any, Tuple
import re
from collections import defaultdict

# Import custom modules
try:
    from src.data_processing.vector_indexer import VectorIndexer
    from src.data_processing.hospital_lookup import HospitalLookup
    from src.pipeline.intent_classifier import TFIDFIntentClassifier
    from src.pipeline.input_processor import InputProcessor
    from src.pipeline.output_formatter import OutputFormatter
    from src.pipeline.inference_engine import InferenceEngine
    print("Custom modules imported successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure the project path is correct and files are accessible")

Custom modules imported successfully!


In [4]:
# Load preprocessed data and indices
def load_data_and_indices(data_path):
    """Load processed data and vector indices."""
    try:

        with open(f"{data_path}/processed_data.json", 'r') as f:
            processed_data = json.load(f)


        vector_indexer = VectorIndexer()
        vector_indexer.load_indices(f"{data_path}/indices")

        print(f"Data loaded: {processed_data['metadata']}")
        return processed_data, vector_indexer

    except Exception as e:
        print(f"Error loading data: {e}")
        return None, None


data_path = "/content/drive/MyDrive/lingo/task1/data"
processed_data, vector_indexer = load_data_and_indices(data_path)


Loading sentence transformer model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Loaded FAQ index with 16412 entries
Loaded tips index with 79 entries
Indices loaded successfully!
Data loaded: {'faq_count': 16412, 'tips_count': 79, 'hospital_faq_count': 7}


In [5]:
# Initialize all components
def initialize_components(processed_data, vector_indexer):
    """Initialize all LingoBot components."""
    try:
        input_processor = InputProcessor()
        intent_classifier = TFIDFIntentClassifier()
        hospital_lookup = HospitalLookup(processed_data['hospital'])
        output_formatter = OutputFormatter()

        print("All components initialized successfully!")
        return input_processor, intent_classifier, hospital_lookup, output_formatter

    except Exception as e:
        print(f"Error initializing components: {e}")
        return None, None, None, None

# Initialize components
input_processor, intent_classifier, hospital_lookup, output_formatter = initialize_components(
    processed_data, vector_indexer
)


All components initialized successfully!


## Qwen2-1.5B-Instruct model

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Inference Engine with Qwen2
class ColabInferenceEngine:
    """
    Qwen2-1.5B-Instruct model for higher quality responses.
    """

    def __init__(self):
        self.model = None
        self.tokenizer = None

    def load_model(self):
        """Load the quantized Qwen2-1.5B-Instruct model."""
        try:
            model_name = "Qwen/Qwen2-1.5B-Instruct"
            print(f"Loading model: {model_name}")

            # Configure 4-bit quantization to save memory
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16
            )

            # Load the tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)

            # Load the model with the quantization config
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto"
            )

            # Set pad token if it's not set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            print("Model loaded successfully!")

        except Exception as e:
            print(f"Error loading model: {e}")
            print("Falling back to rule-based responses...")
            self.model = None
            self.tokenizer = None

    def generate_response(self, query, context, intent):
        """Generate a response using the Qwen2 model and a chat template."""
        if self.model is None or self.tokenizer is None:
            return self._generate_rule_based_response(query, context, intent)

        try:
            # Use the model's chat template for proper instruction formatting
            system_prompt = ""
            user_prompt = f"**Context:**\n{context}\n\n**Question:**\n{query}"

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]

            inputs = self.tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(self.model.device)

            # Generate the response
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs,
                    max_new_tokens=150,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            # 6. Decode the response, skipping the prompt part
            response_ids = outputs[0][inputs.shape[-1]:]
            response = self.tokenizer.decode(response_ids, skip_special_tokens=True).strip()

            return response if response else self._generate_rule_based_response(query, context, intent)

        except Exception as e:
            print(f"Generation error: {e}")
            return self._generate_rule_based_response(query, context, intent)

    def _generate_rule_based_response(self, query, context, intent):
        """Generate rule-based response when model fails."""
        if intent == "General Health FAQ":
            return f"Based on the available information: {context[:200]}..."
        elif intent == "Healthy Lifestyle Tip":
            return f"Here's a helpful tip: {context[:200]}..."
        elif intent == "Hospital Information":
            return f"Hospital information: {context[:200]}..."
        else:
            return "I'm here to help with health questions, lifestyle tips, and hospital information. How can I assist you?"


# Initialize and load the engine
print("Initializing Inference Engine...")
inference_engine = ColabInferenceEngine()
inference_engine.load_model()

# Example usage (after loading)
print("\n--- Running Example Query ---")
example_query = "What are the symptoms of the flu?"
example_context = "Influenza (flu) is a contagious respiratory illness caused by influenza viruses. Symptoms can be mild to severe and commonly include fever, cough, sore throat, runny or stuffy nose, body aches, headache, chills, and fatigue."
example_intent = "General Health FAQ"

if inference_engine.model:
    response = inference_engine.generate_response(example_query, example_context, example_intent)
    print(f"Q: {example_query}")
    print(f"A: {response}")

Initializing Inference Engine...
Loading model: Qwen/Qwen2-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model loaded successfully!

--- Running Example Query ---
Q: What are the symptoms of the flu?
A: The flu typically consists of the following symptoms:

1. Fever or chills.
2. Fatigue.
3. Headache.
4. Muscle or body aches.
5. Dry cough.
6. Shortness of breath.
7. Sore throat.
8. Runny or stuffy nose.
9. New York Times: 10 signs of flu

It's important to note that these symptoms can vary from person to person. The flu virus causes different types of flu infections, including seasonal flu and pandemic flu. Seasonal flu usually occurs in winter and lasts for about two weeks, while pandemic flu can last longer than three months.


## DialoGPT-small

In [16]:
# Inference Engine with DialoGPT-small
class ColabInferenceEngine:

    def __init__(self):
        self.model = None
        self.tokenizer = None

    def load_model(self):
        """Load a lightweight model for Colab."""
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM

            model_name = "microsoft/DialoGPT-small"

            print(f"Loading model: {model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name)

            # Set pad token
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            print("Model loaded successfully!")

        except Exception as e:
            print(f"Error loading model: {e}")
            print("Using rule-based responses instead...")

    def generate_response(self, query, context, intent):
        """Generate response based on context."""
        if self.model is None:
            return self._generate_rule_based_response(query, context, intent)

        try:
            # Simple prompt
            prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

            # Tokenize
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            # Decode
            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = response[len(prompt):].strip()

            return response if response else self._generate_rule_based_response(query, context, intent)

        except Exception as e:
            print(f"Generation error: {e}")
            return self._generate_rule_based_response(query, context, intent)

    def _generate_rule_based_response(self, query, context, intent):
        """Generate rule-based response when model fails."""
        if intent == "General Health FAQ":
            return f"Based on the available information: {context[:200]}..."
        elif intent == "Healthy Lifestyle Tip":
            return f"Here's a helpful tip: {context[:200]}..."
        elif intent == "Hospital Information":
            return f"Hospital information: {context[:200]}..."
        else:
            return "I'm here to help with health questions, lifestyle tips, and hospital information. How can I assist you?"

# Initialize inference engine
inference_engine = ColabInferenceEngine()
inference_engine.load_model()

Loading model: microsoft/DialoGPT-small
✅ Model loaded successfully!


In [7]:
# Main Query Processing Pipeline
class ColabLingoBot:
    """Main LingoBot class."""

    def __init__(self, input_processor, intent_classifier, hospital_lookup,
                 output_formatter, vector_indexer, inference_engine):
        self.input_processor = input_processor
        self.intent_classifier = intent_classifier
        self.hospital_lookup = hospital_lookup
        self.output_formatter = output_formatter
        self.vector_indexer = vector_indexer
        self.inference_engine = inference_engine

    def process_query(self, user_input):
        """Process user query and generate response."""
        try:
            # Process input
            processed_input = self.input_processor.preprocess_query(user_input)

            if not processed_input['is_valid']:
                return processed_input['error_message']

            # Classify intent
            intent, confidence = self.intent_classifier.classify_intent(processed_input['cleaned_query'])

            # Get context based on intent
            context = self._get_context(processed_input['cleaned_query'], intent)

            # Generate response
            if intent == "Greeting or Chit-chat":
                response = self.output_formatter.format_greeting_response(user_input)
            else:
                raw_response = self.inference_engine.generate_response(
                    processed_input['cleaned_query'], context, intent
                )
                response = self.output_formatter.format_response(raw_response)

            return response

        except Exception as e:
            return f"I'm sorry, I encountered an error: {str(e)}"

    def _get_context(self, query, intent):
        """Get relevant context based on intent."""
        context_parts = []

        if intent == "General Health FAQ":
            try:
                faq_results = self.vector_indexer.search_faq(query, top_k=2)
                for result in faq_results:
                    context_parts.append(f"Q: {result['question']}")
                    context_parts.append(f"A: {result['answer']}")
            except:
                context_parts.append("FAQ information not available")

        elif intent == "Healthy Lifestyle Tip":
            try:
                tips_results = self.vector_indexer.search_tips(query, top_k=2)
                for result in tips_results:
                    context_parts.append(f"Tip: {result['tip']}")
            except:
                context_parts.append("Lifestyle tips not available")

        elif intent == "Hospital Information":
            try:
                hospital_results = self.hospital_lookup.lookup(query)
                for result in hospital_results:
                    if result['type'] == 'working_hours':
                        context_parts.append(f"Working Hours: {result['data']}")
                    elif result['type'] == 'contact':
                        context_parts.append(f"Contact: {result['data']}")
                    elif result['type'] == 'address':
                        context_parts.append(f"Address: {result['data']}")
                    elif result['type'] == 'appointment':
                        context_parts.append(f"Appointment: {result['data']}")
                    elif result['type'] == 'departments':
                        context_parts.append(f"Departments: {result['data']}")
            except:
                context_parts.append("Hospital information not available")

        return " ".join(context_parts) if context_parts else "No relevant information found."

# Initialize LingoBot
lingobot = ColabLingoBot(
    input_processor, intent_classifier, hospital_lookup,
    output_formatter, vector_indexer, inference_engine
)

print("LingoBot initialized and ready!")


LingoBot initialized and ready!


In [ ]:
# Simple Text-based Chat
def simple_chat():
    """Simple text-based chat interface."""
    print("🤖 LingoBot Healthcare Assistant")
    print("Type 'quit' to exit\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("👋 Goodbye!")
            break

        response = lingobot.process_query(user_input)
        print(f"LingoBot: {response}\n")

# Uncomment to run simple chat
# simple_chat()


🤖 LingoBot Healthcare Assistant
Type 'quit' to exit

You: how to reduce weight?
Generation error: name 'torch' is not defined
LingoBot: I'm here to help with health questions, lifestyle tips, and hospital information. How can I assist you.

You: hello
LingoBot: Good day! I'm LingoBot. I can help you with health FAQs, wellness tips, or hospital information. What can I assist you with?

Disclaimer: This is for educational purposes only and not a substitute for professional medical advice.

You: how to reduce weight?
Generation error: name 'torch' is not defined
LingoBot: I'm here to help with health questions, lifestyle tips, and hospital information. How can I assist you.

You: quit
👋 Goodbye!


In [24]:
# Gradio Interface
import gradio as gr

def gradio_chat_interface():
    """Create Gradio interface for LingoBot."""

    def chat_function(message, history):
        """Process chat message and return response."""
        response = lingobot.process_query(message)
        # Add disclaimer to every response
        disclaimer = "\n\n⚠️Disclaimer: This is for educational purposes only and not a substitute for professional medical advice. Always consult a healthcare professional for personalized medical guidance."

        return response + disclaimer

    # Create Gradio interface
    interface = gr.ChatInterface(
        fn=chat_function,
        title="🤖 LingoBot Healthcare Assistant",
        description="Ask me about health FAQs, lifestyle tips, or hospital information!",
        examples=[
            "What are the symptoms of flu?",
            "How much sleep should I get?",
            "What are the hospital's working hours?",
            "Give me tips for healthy eating",
            "How to book an appointment?"
        ],
        chatbot=gr.Chatbot(height=400),
        textbox=gr.Textbox(
            placeholder="Ask me anything about health...",
            container=False,
            scale=7
        ),
        theme=gr.themes.Soft(),
        css="""
          .gradio-container {
          font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
          width: 95vw !important; /* Use 95% of viewport width */
          height: 95vh !important; /* Use 95% of viewport height */
        }
            .disclaimer {
            font-size: 12px;
            color: #888;
            text-align: center;
            margin-top: 20px;
            padding: 10px;
            background-color: #f8f9fa;
            border-radius: 5px;
        }
        """
    )

    return interface

# Create and launch Gradio interface
gradio_interface = gradio_chat_interface()
gradio_interface.launch(share=True, debug=True)

/tmp/ipython-input-4082403462.py:27: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=400),
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:328: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d161321f86242247c8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d161321f86242247c8.gradio.live


In [26]:
# IPython Widgets Interface (Alternative to Gradio)
import ipywidgets as widgets
from IPython.display import display, clear_output

def create_widget_interface():
    """Create IPython widgets interface."""

    # Create widgets
    text_input = widgets.Textarea(
        value='',
        placeholder='Ask me anything about health...',
        description='Question:',
        layout=widgets.Layout(width='100%', height='80px')
    )

    send_button = widgets.Button(
        description='Ask LingoBot',
        button_style='primary',
        layout=widgets.Layout(width='150px')
    )

    output_area = widgets.Output()

    def on_send_clicked(b):
        """Handle send button click."""
        with output_area:
            clear_output(wait=True)
            if text_input.value.strip():
                print("🤖 LingoBot is thinking...")
                response = lingobot.process_query(text_input.value)
                print(f"🤖 LingoBot: {response}")
                print("\n" + "="*50 + "\n")
            else:
                print("Please enter a question!")

    # Connect button to function
    send_button.on_click(on_send_clicked)

    # Create interface
    interface = widgets.VBox([
        widgets.HTML("<h2>🤖 LingoBot Healthcare Assistant</h2>"),
        widgets.HTML("<p>Ask me about health FAQs, lifestyle tips, or hospital information!</p>"),
        text_input,
        send_button,
        output_area
    ])

    return interface

# Uncomment to use widget interface
widget_interface = create_widget_interface()
display(widget_interface)


## Usage Instructions

1. **Update Paths**: Modify the `project_path` and `data_path` variables to point to your Google Drive location
2. **Run Cells**: Execute all cells in order to initialize the system
3. **Choose Interface**:
   - **Gradio** (Recommended): Provides a web interface with sharing capability
   - **Widgets**: Native Jupyter interface
   - **Text Chat**: Simple command-line style chat
4. **Option**: either use Qwen2-1.5B-Instruct or DialoGPT-small for inference   engine.
## Important Notes

- Make sure your pre-computed indices and data are accessible in Google Drive
- Gradio interface provides a public URL for sharing
